In [1]:
import numpy as np
import scanpy as sc
from sklearn.metrics import normalized_mutual_info_score

In [2]:
adata = sc.read_h5ad("/data2/a330d/datasets/crc/processed/crc_cosmx_wt.h5ad")

In [3]:
# Remove cells with nan jst values
adata = adata[~adata.obs['jst'].isna()]

In [14]:
nmi_very_fine = normalized_mutual_info_score(
    adata.obs['jst'], adata.obs['typ_clean']
)

In [15]:
nmi_fine = normalized_mutual_info_score(
    adata.obs['ist'], adata.obs['typ_clean']
)

In [16]:
nmi_coarse = normalized_mutual_info_score(
    adata.obs['coarse_type'], adata.obs['typ_clean']
)

In [17]:
# Print in a nicer format, 2 decimal places
print(f"{nmi_very_fine:.2f}, {nmi_fine:.2f}, {nmi_coarse:.2f}")

0.14, 0.15, 0.06


In [9]:
for slide in adata.obs.sid.unique():
    if slide == 110:
        continue
    adata_slide = adata[adata.obs.sid == slide]
    # Keep cells that have only 'REF' or 'CRC' in typ_clean column
    adata_slide = adata_slide[adata_slide.obs['typ_clean'].isin(['REF', 'CRC'])]
    adata_slide = adata_slide[~adata_slide.obs['jst'].isna()]
    nmi_fine_slide = normalized_mutual_info_score(
        adata_slide.obs['ist'], adata_slide.obs['typ_clean']
    )
    nmi_coarse_slide = normalized_mutual_info_score(
        adata_slide.obs['coarse_type'], adata_slide.obs['typ_clean']
    )
    print(f"Slide {slide}: NMI fine-grained y: {nmi_fine_slide:.2f}, NMI coarse y: {nmi_coarse_slide:.2f}")

Slide 120: NMI fine-grained y: 0.07, NMI coarse y: 0.01
Slide 210: NMI fine-grained y: 0.12, NMI coarse y: 0.01
Slide 221: NMI fine-grained y: 0.06, NMI coarse y: 0.01
Slide 231: NMI fine-grained y: 0.09, NMI coarse y: 0.03
Slide 232: NMI fine-grained y: 0.11, NMI coarse y: 0.04
Slide 242: NMI fine-grained y: 0.27, NMI coarse y: 0.06


# Cross-check patients

In [ ]:
adata.obs.ist.value_counts()

ist
epi2      913886
epi4      441718
epi3      319102
fib1      223272
epi1      208038
TC        173917
fib2      157263
PC_IgA     87163
mye1       80842
EC         72453
mye2       72231
SMC        71615
PC_IgG     69809
BC         45485
PC_IgM     22448
mast        5528
Name: count, dtype: int64

In [22]:
import pandas as pd

# crosstab of celltype x slide, values = counts
ct = pd.crosstab(adata.obs['ist'], adata.obs['sid'])

THRESHOLD = 1000

# True where a celltype falls below threshold on a given slide
below_thresh = ct < THRESHOLD

# celltypes that fall below threshold on at least one slide
missing_mask = below_thresh.any(axis=1)

n_missing = missing_mask.sum()
celltypes_missing = ct.index[missing_mask].tolist()

print(f"{n_missing} out of {ct.shape[0]} celltypes fall below {THRESHOLD} cells on at least one slide")
print(celltypes_missing)

3 out of 16 celltypes fall below 1000 cells on at least one slide
['BC', 'mast', 'mye2']


In [14]:
missing_detail = {
    ct_name: ct.columns[ct.loc[ct_name] == 0].tolist()
    for ct_name in celltypes_missing
}
missing_detail

{'Bn': [232],
 'cDC2_CCR7': [232],
 'epi.1': [232],
 'epi.paneth-like': [221, 232],
 'epi_DNMT1': [231, 232],
 'epi_LEFTY1': [232],
 'epi_REG/DUOX': [232],
 'fib.TLS': [221],
 'fib_PI16': [221],
 'macro2': [231, 232],
 'mast': [221, 231, 232],
 'nphil': [110]}